In [ ]:
# 导入库以及设置路径
import os
from collections import namedtuple

import pyrootutils as L
from tqdm import tqdm

L.setup_root("..", pythonpath=True)

分割pdbqt文件，每个文件只包含一个分子。

In [ ]:
# 导入自定义包
from src.dock_utils.utils import ZincPdbqt, write_pdbqt_to_gz

In [ ]:
# 设定输入输出路径
src_pdbqt_file = "../data/ligand/zinc20_druglike_random_sample_molecule_1f600_10k.pdbqt.gz"
dest_pdbqt_folder = "../data/ligand/10k_mol"
conformer_folder = "../data/ligand/10k_mol_10_conformer/"

In [ ]:
# 载入测试分子
mol_data_10k = ZincPdbqt(src_pdbqt_file)

In [ ]:
# 分割后的pdbqt分子保存到单个文件当中
for mol_data in mol_data_10k:
    zinc_id, pdbqt_str = mol_data
    with open(dest_pdbqt_folder + "/" + zinc_id + ".pdbqt", "w") as f:
        f.write(pdbqt_str)
    # break

需要针对每一组构象重新生成分子图，每个图单独进行预测。因为模型的缘故，还要使用dataloader。

In [ ]:
# 设定构象的文件路径
file_list = os.listdir(conformer_folder)
len(file_list)

In [ ]:
# 将所有构象读入，并提取构象坐标


def get_mol_pos(file_path_list):
    mols = []
    for file_path in tqdm(file_path_list):
        # file_path = os.path.join(conformer_folder, file)
        # print(file_path)
        conformer_list = []
        with open(file_path, "r") as f:
            MolPos = namedtuple("MolPos", "atom xyz")
            molpos = MolPos([], [])
            for line in f:
                if line.startswith("REMARK  Name"):
                    # 一个新的构象
                    if molpos.atom:
                        # molpos非空，加入构象
                        conformer_list.append(molpos)
                        # 置空
                        molpos = MolPos([], [])
                elif line.startswith("ATOM"):
                    molpos.atom.append(line[12:14].strip())
                    molpos.xyz.append(
                        [
                            float(line[30:38]),
                            float(line[38:46]),
                            float(line[46:54]),
                        ]
                    )
            conformer_list.append(molpos)
        mols.append(conformer_list)
    return mols

In [ ]:
mols = get_mol_pos(file_list)
len(mols)

In [ ]:
# 构造分子图
import torch
from scipy import spatial
from torch_geometric.data import Data, Dataset

elements_dict = dict(C=0, N=1, O=2, H=3, F=4, S=5, CL=6, BR=7, I=8, SI=9, P=10)


def create_graph(molpos):
    # 构建全连接图的edge_index
    edge_index = [[], []]  # 初始化一个二维列表作为edge_index，当前只有一个空列表
    mol_size = len(molpos.atom)  # 获取分子中原子的数量
    for i in range(mol_size):
        edge_index[0].extend([i] * mol_size)  # 在第一维列表中，将i重复molsize次添加到列表中
        edge_index[1].extend(list(range(mol_size)))  # 在第二维列表中，将0到molsize-1的整数添加到列表中
    edge_index = torch.tensor(
        edge_index, dtype=torch.long
    )  # 将edge_index转换为torch张量，数据类型为torch.long
    x = [elements_dict[atom] for atom in molpos.atom]  # 根据原子的元素类型从元素字典中获取对应的元素数值
    distance_matrix = spatial.distance_matrix(molpos.xyz, molpos.xyz)  # 计算原子之间的距离矩阵
    edge_attr = torch.tensor(distance_matrix, dtype=torch.float32).view(
        -1, 1
    )  # 将距离矩阵转换为torch张量，数据类型为torch.float
    return Data(
        x=torch.tensor(x, dtype=torch.long),  # 将元素张量x转换为torch张量，数据类型为torch.long
        edge_index=edge_index,  # 将构建的edge_index作为参数传递给Data类
        edge_attr=edge_attr,  # 将构建的edge_attr作为参数传递给Data类
        pos=torch.tensor(molpos.xyz, dtype=torch.float),  # 将xyz坐标转换为torch张量，数据类型为torch.float
    )


class MolDataset(Dataset):
    def __init__(self, mols):
        super().__init__()
        self.mols = mols

    def len(self):
        """
        返回数据集的长度

        Returns:
            int: 数据集的长度
        """
        return len(self.mols)

    def get(self, idx):
        """
        根据给定的索引获取数据集中的数据

        Args:
            idx (int): 索引值

        Returns:
            object: 数据集中的数据
        """
        return create_graph(self.mols[idx])

## EGNN

In [ ]:
# 10k data model ckpt path
ckpt_path_3a6p = "/home/huabei/project/SMTarRNA/logs/logs/3a6p_100k_opti/multiruns/2023-08-09_18-06-33/16/checkpoints/epoch_088.ckpt"
ckpt_path_4z4c = "/home/huabei/project/SMTarRNA/logs/logs/4z4c_100k_opti/multiruns/2023-08-10_17-08-01/17/checkpoints/epoch_087.ckpt"
ckpt_path_4z4d = "/home/huabei/project/SMTarRNA/logs/logs/4z4d_100k_opti/multiruns/2023-08-09_17-18-36/19/checkpoints/epoch_089.ckpt"
ckpt_path_6cbd = "/home/huabei/project/SMTarRNA/logs/logs/6cbd_100k_opti/multiruns/2023-08-23_19-21-46/11/checkpoints/epoch_082.ckpt"

In [ ]:
from src.models.components.egnn import Egnn
from src.models.smtr_module import SMTARRNAModule

ckpt_path_3a6p = "/home/huabei/project/SMTarRNA/logs/logs/3a6p_total_data_opti/multiruns/2023-08-12_10-44-26/8/checkpoints/epoch_032.ckpt"
ckpt_path_4z4c = "/home/huabei/project/SMTarRNA/logs/logs/4z4c_total_data_opti/multiruns/2023-08-12_10-44-26/12/checkpoints/epoch_059.ckpt"
ckpt_path_4z4d = "/home/huabei/project/SMTarRNA/logs/logs/4z4d_total_data_opti/multiruns/2023-08-12_10-44-26/14/checkpoints/epoch_050.ckpt"
ckpt_path_6cbd = "/home/huabei/project/SMTarRNA/logs/logs/6cbd_total_data_opti/multiruns/2023-08-23_19-33-11/3/checkpoints/epoch_089.ckpt"


# 设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = Egnn(in_node_nf=11, hidden_nf=128, out_node_nf=5, n_layers=3, attention=True, normalize=True)
# 实例化pl模型
# model = SMTARRNAModule(net, None)
# 载入参数
# model.load_form_checkpoint(ckpt_path)
model = SMTARRNAModule(net, None).to(device)
# model.eval()

In [ ]:
paras = [ckpt_path_3a6p, ckpt_path_4z4c, ckpt_path_4z4d, ckpt_path_6cbd]

## molecule GNN

In [ ]:
ckpt_path_3a6p_molecule_gnn = "/home/huabei/project/SMTarRNA/logs/molecule_gnn_retrain/runs/2023-11-15_21-41-11/checkpoints/last.ckpt"
from src.models.components.molecule_gnn import MoleculeGnn
from src.models.smtr_module import SMTARRNAModule

# 设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

net = MoleculeGnn(in_node_nf=11, hidden_nf=256, out_node_nf=5, n_layers=8, out_layers=6)
# 实例化pl模型
# model = SMTARRNAModule(net, None)
# 载入参数
# model.load_form_checkpoint(ckpt_path)
model = SMTARRNAModule(net, None).to(device)
model.load_state_dict(torch.load(ckpt_path_3a6p_molecule_gnn)["state_dict"])

In [ ]:
import numpy as np
from torch_geometric.loader import DataLoader


def get_comformer_var(model):
    """给出模型在数据集各个分子10个conformer上的方差的平均值"""
    with torch.no_grad():
        total_mol_energy_var = []
        for mol_conformers in tqdm(mols):
            # 计算每个分子不同构象能量之间的方差
            mol_energy = []
            for g in DataLoader(MolDataset(mol_conformers), batch_size=10):  # 只会有一个batch
                mol_energy.append(model.forward(g.to(device)).cpu().numpy())  # 一个元素的list
            total_mol_energy_var.append(np.var(mol_energy[0], axis=0))
        # return np.mean(np.array(total_mol_energy_var), axis=0)
        return np.array(total_mol_energy_var)  # N * 5


# main流程
complex_conformer_var = []
for ckpt_path in paras:
    # 载入模型
    model.load_state_dict(torch.load(ckpt_path)["state_dict"])
    model.eval()
    complex_conformer_var.append(get_comformer_var(model))

# print(complex_conformer_var_mean)

In [ ]:
import numpy as np
from torch_geometric.loader import DataLoader

with torch.no_grad():
    total_mol_energy_var = []
    for mol_conformers in tqdm(mols):
        mol_energy = []
        for g in DataLoader(MolDataset(mol_conformers), batch_size=10):
            mol_energy.append(model.forward(g.to(device)).cpu().numpy())
        total_mol_energy_var.append(np.var(mol_energy[0], axis=0))

        # break

In [ ]:
# 计算每个分子10个构象预测能量之间的方差
import numpy as np

mol_conformer_energy_var = []
for mol_e in total_mol_energy:
    mol_conformer_energy_var.append(np.var(torch.concat(mol_e, dim=0).numpy(), axis=0))

# 绘图

In [ ]:
x = ["3a6p", "4z4c", "4z4d", "6cbd"]

In [ ]:
y = [i[:, 0] for i in complex_conformer_var]  # 取出第一列
y[0].shape

In [ ]:
y = np.array(y)
y.shape

In [ ]:
# 保存y
import pickle

with open("energy_var_y.pkl", "wb") as f:
    pickle.dump(y, f)

In [ ]:
# 导入y
import pickle

with open("energy_var_y.pkl", "rb") as f:
    y = pickle.load(f)

In [ ]:
for i in range(4):
    print((y[i] > 0.5).sum() / 10000)

In [ ]:
# 绘制一个条形图
# import matplotlib.pyplot as plt
import seaborn as sns

### molecule GNN

In [ ]:
import matplotlib.pyplot as plt

fig = sns.violinplot(data=y.T)
plt.xlabel("molecule gnn Model", fontsize=12)
plt.ylabel("Variance", fontsize=12)
plt.xticks([0], ["molecule_gnn"], fontsize=10)
# plt.show()
# 保存图片
fig.get_figure().savefig("molecule_gnn_variance_10k_.png", dpi=300)

### EGNN

In [ ]:
# y = np.array(total_mol_energy_var)[:, 0]
y.T.shape

In [ ]:
import matplotlib.pyplot as plt
from pyrsistent import b

fig = plt.figure(figsize=(3.5, 2.5), dpi=300)
# 绘制小提琴图
sns.violinplot(data=y.T, linewidth=0.7, width=0.5, cut=0, palette="Set2")
plt.xlabel("Complex", fontsize=8)
plt.ylabel("Variance", fontsize=8)
plt.xticks([0, 1, 2, 3], x, fontsize=6)
# 横向网格线
plt.grid(axis="y", linestyle="--", linewidth=0.5)
# 刻度线
plt.tick_params(axis="both", labelsize=6, width=0.5, length=1.5)
# plt.show()
plt.rcParams["font.family"] = "Times New Roman"
# 保存图片
plt.savefig("fig/variance.svg", bbox_inches="tight")
plt.savefig("fig/variance.png", bbox_inches="tight")

In [ ]:
import os

os.getcwd()

## 对分子进行聚类分析
依据方差作为聚类点的颜色

In [ ]:
import os

# smi folder
smi_folder = "../data/ligand/10k_mol_smi/"
# 读取smi文件，保存到dict中
smi_dict = {}
for file in os.listdir(smi_folder):
    file_path = os.path.join(smi_folder, file)
    with open(file_path, "r") as f:
        # print(f.readline())
        smi, zinc_id = tuple(f.readline().strip().split(" "))
        smi_dict[zinc_id] = smi
    # break

In [ ]:
# 下载smi文件
ligand_folder = "../data/ligand/10k_mol/"
zinc_id_list = []
for file in os.listdir(ligand_folder):
    zinc_id_list.append(file.split(".")[0])
# zinc_id_list = list(smi_dict.keys())
from time import sleep

import requests
from tqdm import tqdm


def create_smiles_url(zinc_id: str):
    return "https://zinc20.docking.org/substances/" + zinc_id + ".smi"


for zinc_id in tqdm(zinc_id_list):
    try:
        smiles = requests.get(create_smiles_url(zinc_id))
    except:
        print(zinc_id)
        continue
    smiles = str(smiles.content, encoding="utf-8")
    with open(smi_folder + zinc_id + ".smi", "w") as f:
        f.write(smiles)
    # sleep(0.1)

In [ ]:
import dis

import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA

# 输入SMILES列表
# smiles_list = ['CC', 'CCO', 'CCC', 'CCN', 'CCCN', 'CCCO', 'CCCC']
smiles_list = smi_list
# smiles_list = list(smi_dict.values())

# 从SMILES创建分子
mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# 计算Morgan fingerprints
fps = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, 1024) for mol in mols]
dis_matrix = [
    DataStructs.BulkTanimotoSimilarity(fps[i], fps, returnDistance=True) for i in range(10000)
]
dis_array = np.array(dis_matrix)
# fps = np.array([list(fp) for fp in fps])

In [ ]:
# 使用PCA进行降维
pca = PCA(n_components=2)
pca.fit(dis_array)

In [ ]:
dis_pca = pca.transform(dis_array)

In [ ]:
# 使用kmeans进行聚类
kmeans = KMeans(n_clusters=6)
kmeans.fit(dis_array)
# ward = AgglomerativeClustering(n_clusters=6)
# ward.fit(dis_array)

In [ ]:
# 绘制四个子图
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    # 绘制聚类结果
    ax.scatter(dis_pca[:, 0], dis_pca[:, 1], c=y[i], cmap="Paired", s=2, alpha=0.5)
    # 添加title
    ax.set_title(f"{x[i]}")

# 添加色标
cax = fig.add_axes([0.92, 0.1, 0.03, 0.8])
cax.set_title("Variance")
fig.colorbar(axes[1, 1].collections[0], cax=cax)
# 绘制聚类结果
# plt.scatter(dis_pca[:, 0], dis_pca[:, 1], c=y[3], cmap='Paired', s=2)
# # plt.scatter(dis_pca[:, 0], dis_pca[:, 1], c=ward.labels_)
# # 添加色标
# plt.colorbar()

# 保存图片
plt.savefig("10k_molecule_conformer_var_vs_pca.png", dpi=300)

# plt.show()

In [ ]:
import os

conformer_folder = "../data/ligand/10k_mol_10_conformer/"
file_list = os.listdir(conformer_folder)
zinc_id_list = [file.split(".")[0][-16:] for file in file_list]
# 将smi的顺序和conformer var的顺序对应起来
smi_list = []
for k in zinc_id_list:
    smi_list.append(smi_dict[k])

## 计算构像差异

In [ ]:
pdbqt_conformer_folder = "../data/ligand/10k_mol_10_conformer/"
out_folder = "../data/ligand/10k_mol_10_conformer_pdbqt/"
file_list = os.listdir(pdbqt_conformer_folder)
for file in file_list:
    file_path = os.path.join(pdbqt_conformer_folder, file)
    with open(file_path, "r") as f:
        total_rows = f.readlines()
        # 分割为10个构象
        length_ = len(total_rows)
        single_file_length = length_ // 10
        # 保存到单独的文件中
        with open(os.path.join(out_folder, file), "w") as f:
            for i in range(10):
                f.write("MODEL " + str(i) + "\n")
                f.writelines(total_rows[i * single_file_length : (i + 1) * single_file_length])
                f.write("ENDMDL\n")
    # break

In [ ]:
src_folder = "../data/ligand/10k_mol/"
conformer_folder = "../data/ligand/10k_mol_10_conformer/"

In [ ]:
base_file_list = os.listdir(src_folder)
base_file_list.sort()
base_file_list = [os.path.join(src_folder, file) for file in base_file_list]
conformer_file_list = os.listdir(conformer_folder)
conformer_file_list.sort()
conformer_file_list = [os.path.join(conformer_folder, file) for file in conformer_file_list]
base_mols = get_mol_pos(base_file_list)
conformer_mols = get_mol_pos(conformer_file_list)

In [ ]:
# 验证顺序是否一致
conformer_file_list[:10], base_file_list[:10]

In [ ]:
import numpy as np
from tqdm import tqdm

rmsd_list = []
for i in range(len(base_file_list)):
    # print(conformer_file_list[i])
    conformer_pos = []
    for j in range(len(conformer_mols[i])):
        # print(len(conformer_mols[i]))
        conformer_pos.append(conformer_mols[i][j].xyz)
    conformer_pos = np.array(conformer_pos)
    base_pos = np.array(base_mols[i][0].xyz)
    # 计算rmsd
    rmsd = np.sqrt(np.mean(((conformer_pos - base_pos[None, :, :]) ** 2).sum(axis=-1), axis=1))
    rmsd_list.append(rmsd)
    # break

In [ ]:
conformer_pos.shape, base_pos[None, :, :].shape, rmsd.mean()

In [ ]:
rmsd_mean_list = [i.mean() for i in rmsd_list]
rmsd_var_list = [i.var() for i in rmsd_list]

In [ ]:
rmsd_mean_array = np.array(rmsd_mean_list)
rmsd_var_array = np.array(rmsd_var_list)
rmsd_mean_array.mean()

In [ ]:
fig = sns.violinplot(data=rmsd_mean_array)

In [ ]:
fig = sns.violinplot(data=rmsd_var_array)

In [ ]:
x_e = np.array([[1, 2, 3, 4]] * 10000)
x_e = x_e.T.reshape(-1, 1)[:, 0]
y_e = y.reshape(-1, 1)[:, 0]
x_e.shape, y_e.shape

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

# 以rmsd的均值小提琴图作为背景，绘制y的小提琴图
fig, ax = plt.subplots(figsize=(3.5, 2.5), dpi=300)
# rmsd使用右侧y轴
# ax2 = ax.twinx()
# 绘制y的小提琴图
sns.violinplot(
    x=x_e, y=y_e, ax=ax, linewidth=0.3, width=0.7, cut=0, palette="Set2", saturation=0.5
)
# 绘制rmsd的小提琴图, 嵌入到y的小提琴图中
inset_ax = fig.add_axes([0.2, 0.5, 0.3, 0.3])
parts = inset_ax.violinplot(rmsd_mean_array, positions=[1.5], widths=0.5, showextrema=False)
# for pc in parts['bodies']:
#     pc.set_facecolor('orangered')
# pc.set_edgecolor('black')
# pc.set_alpha(0.3)
# 设置title和图的距离
inset_ax.set_title("RMSD distribution", fontsize=6, pad=2)

inset_ax.set_xticks([])
inset_ax.set_ylabel("RMSD(Å)", fontsize=4, labelpad=0.1)
inset_ax.set_yticks([0, 2, 4, 6, 8, 10])
# 字体大小
inset_ax.tick_params(axis="both", labelsize=5, width=0.5, length=1.2)


# 设置x轴刻度
ax.set_xticks(
    [1, 2, 3, 4],
)
# ax.set_xlim(0, 3.5)
ax.set_xlabel("Complex", fontsize=6)
ax.set_ylabel("Variance", fontsize=6)
ax.set_xticks([0, 1, 2, 3], ["3a6p", "4z4c", "4z4d", "6cbd"], fontsize=6)
# 横向网格线
ax.grid(axis="y", linestyle="--", linewidth=0.3)
# 刻度线
ax.tick_params(axis="both", labelsize=6, width=0.5, length=2)
# plt.show()
plt.rcParams["font.family"] = "Times New Roman"
# 右侧y轴
# ax2.set_ylabel('RMSD', fontsize=6)
# ax2.tick_params(axis='y', labelsize=6, width=0.5, length=1.5)
# # 一位小数
# ax2.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

# ax2.set_ylim(-0.4, 15)

ax.set_xlim(-0.7, 3.5)
# # 保存图片
plt.savefig("fig/conformer_variance_and _rmsd.svg", bbox_inches="tight")
plt.savefig("fig/conformer_variance_and _rmsd.png", bbox_inches="tight")

In [ ]:
# rmsd = []
# e = 0
# for i in range(10000):
#     # print(src_files[i], conformer_files[i])
#     base_conf = Chem.SDMolSupplier(src_folder + src_files[i])[0]
#     try:
#         # conformer_mols = Chem.SDMolSupplier(conformer_folder + conformer_files[i])
#         # pdb
#         conformer_mols = Chem.MolFromPDBFile(conformer_folder + conformer_files[i])
#     except:
#         print(src_files[i], conformer_files[i])
#         continue
#     # 计算rmsd
#     rmsd_list = []
#     for conformer_mol in conformer_mols:
#         # print('ok')
#         try:
#             rmsd_list.append(AllChem.GetBestRMS(base_conf, conformer_mol))
#         except:
#             e += 1
#             pass
#     rmsd.append(rmsd_list)
#     # break

In [ ]:
# 绘制分子
y[3, :][y[3, :] > 2].shape